# 11 – Kiểm định thống kê bổ sung (theo phản biện)

Notebook này **không train lại mô hình**. Nó dùng dự đoán và trọng số đã lưu ở notebook 02 và 10 để trả lời các câu hỏi của phản biện:

1. **Bootstrap theo người nói (cluster bootstrap):** khoảng tin cậy có bị hẹp quá do nhiều câu cùng một người nói không?
2. **Biến thiên theo seed:** khác biệt Geo-LS và LS trên mô hình kết hợp có ổn định qua từng seed không?
3. **Hiệu chuẩn (calibration):** ECE, Brier, NLL thay vì chỉ NLL.
4. **Không thua kém (non-inferiority):** Geo-LS có thực sự "không mất accuracy" không, với biên nào?
5. **Kiểm định Mantel/QAP:** tương quan giữa nhầm lẫn và khoảng cách, với giả thuyết không đúng cho dữ liệu cặp tỉnh phụ thuộc nhau.
6. **Giới tính:** phân bố nam/nữ theo tỉnh, giới hạn trên của việc đoán tỉnh chỉ từ giới tính, accuracy theo giới tính.
7. **Tập con kiểm soát chặt hơn:** cả TF-IDF và PhoBERT đều sai; cả LTAS và MFCC đều sai.

- **Runtime:** CPU là đủ cho mục 1, 3–7 (vài phút). Mục 2 cần GPU (T4/L4) để nạp đặc trưng, khoảng 5 phút; đặt `RUN_SEEDS = False` để bỏ qua.
- **Cần đã chạy:** notebook 01, 02, 10 (thư mục `results/final/` có `test_probs_*.npy` và `head_D_*_seed*.pt`).
- **Kết quả:** `results/audit/audit_report.json` và 2 hình. Gửi file JSON cho Claude.

In [ ]:
# ==== CẤU HÌNH ====
DRIVE_ROOT = "/content/drive/MyDrive/vimd_dl"
PROVINCE_COL = "province_code"
FEAT_B = "full13L_wav2vec2-xls-r-300m_12s"
FEAT_C = "enc_PhoWhisper-large"
SEEDS = [0, 1, 2]
RUN_SEEDS = True          # mục 2: cần GPU và đặc trưng trên Drive
N_BOOT = 2000             # số lần bootstrap
N_PERM = 9999             # số hoán vị cho kiểm định Mantel/QAP
NI_MARGINS = [0.01, 0.015, 0.02]   # biên không-thua-kém / tương đương (theo tỉ lệ, 0.01 = 1 điểm %)

In [ ]:
import os, json, time, warnings
import numpy as np, pandas as pd
from scipy.stats import spearmanr, rankdata, chi2_contingency
from sklearn.metrics import f1_score
warnings.filterwarnings("ignore")
T0 = time.time()
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
RES_DIR = f"{DRIVE_ROOT}/results"; FIN = f"{RES_DIR}/final"; OUT = f"{RES_DIR}/audit"
os.makedirs(OUT, exist_ok=True)

meta = pd.read_csv(f"{DRIVE_ROOT}/metadata.csv", low_memory=False)
meta[PROVINCE_COL] = meta[PROVINCE_COL].astype(str)
prov = pd.read_csv(f"{DRIVE_ROOT}/provinces.csv"); prov[PROVINCE_COL] = prov[PROVINCE_COL].astype(str)
CLASSES = sorted(meta[PROVINCE_COL].unique()); C = len(CLASSES)
y = meta[PROVINCE_COL].map({c: i for i, c in enumerate(CLASSES)}).values
use = ~meta["exclude"].astype(bool).values
tr, va, te = [(meta["split"].values == s) & use for s in ("train", "valid", "test")]
pidx = prov.set_index(PROVINCE_COL)
NAMES = [pidx.loc[c, "province_name"] for c in CLASSES]
region_of = np.array([pidx.loc[c, "region"] for c in CLASSES])
yt = y[te]; n_te = len(yt)

COORD = {
 "CaoBang": (22.666, 106.258), "LangSon": (21.853, 106.761), "QuangNinh": (20.951, 107.080), "HaiPhong": (20.845, 106.688),
 "ThaiBinh": (20.446, 106.336), "NamDinh": (20.420, 106.168), "PhuTho": (21.322, 105.402), "ThaiNguyen": (21.594, 105.848),
 "YenBai": (21.705, 104.875), "TuyenQuang": (21.823, 105.214), "HaGiang": (22.823, 104.984), "LaoCai": (22.486, 103.971),
 "LaiChau": (22.396, 103.458), "SonLa": (21.326, 103.917), "DienBien": (21.386, 103.023), "HoaBinh": (20.817, 105.338),
 "HaNoi": (21.028, 105.854), "HaiDuong": (20.940, 106.333), "NinhBinh": (20.251, 105.975), "VinhPhuc": (21.310, 105.597),
 "HungYen": (20.646, 106.051), "HaNam": (20.541, 105.914), "BacKan": (22.147, 105.835), "BacGiang": (21.273, 106.195),
 "BacNinh": (21.186, 106.076), "ThanhHoa": (19.807, 105.776), "NgheAn": (18.679, 105.682), "HaTinh": (18.343, 105.906),
 "QuangBinh": (17.468, 106.622), "QuangTri": (16.816, 107.100), "ThuaThienHue": (16.464, 107.586), "DaNang": (16.054, 108.202),
 "QuangNam": (15.573, 108.474), "QuangNgai": (15.120, 108.792), "BinhDinh": (13.776, 109.224), "PhuYen": (13.096, 109.309),
 "KhanhHoa": (12.239, 109.197), "NinhThuan": (11.565, 108.988), "BinhThuan": (10.928, 108.102), "KonTum": (14.350, 108.000),
 "GiaLai": (13.983, 108.000), "DakLak": (12.667, 108.038), "DakNong": (12.004, 107.690), "LamDong": (11.940, 108.458),
 "HoChiMinh": (10.776, 106.701), "DongNai": (10.957, 106.843), "BinhDuong": (10.980, 106.651), "BaRiaVungTau": (10.496, 107.169),
 "TayNinh": (11.310, 106.098), "BinhPhuoc": (11.535, 106.883), "LongAn": (10.535, 106.413), "TienGiang": (10.360, 106.360),
 "BenTre": (10.241, 106.376), "VinhLong": (10.254, 105.972), "TraVinh": (9.935, 106.345), "CanTho": (10.045, 105.747),
 "HauGiang": (9.784, 105.470), "SocTrang": (9.603, 105.980), "BacLieu": (9.294, 105.727), "CaMau": (9.176, 105.150),
 "DongThap": (10.460, 105.633), "AnGiang": (10.386, 105.435), "KienGiang": (10.012, 105.081),
}
LAT = np.array([COORD[n][0] for n in NAMES]); LON = np.array([COORD[n][1] for n in NAMES])
p_ = np.pi / 180
DIST = 2 * 6371 * np.arcsin(np.sqrt(np.sin((LAT[None] - LAT[:, None]) * p_ / 2) ** 2 + np.cos(LAT[:, None] * p_)
                                    * np.cos(LAT[None] * p_) * np.sin((LON[None] - LON[:, None]) * p_ / 2) ** 2))

# xác suất trên tập test (ensemble 3 seed, lưu ở notebook 10) và dự đoán của baseline (notebook 02)
PROBS = {
    "B_xlsr|ls": np.load(f"{FIN}/test_probs_B_xlsr_ls.npy"),
    "C_phowhisper|ls": np.load(f"{FIN}/test_probs_C_phowhisper_ls.npy"),
    "D_fusion|ls": np.load(f"{FIN}/test_probs_D_fusion_ls.npy"),
    "D_fusion|geols": np.load(f"{FIN}/test_probs_D_fusion_geols.npy"),
    "ensemble_BCD|ls": np.load(f"{FIN}/test_probs_FINAL_acc.npy"),
}
PRED = {k: v.argmax(1) for k, v in PROBS.items()}
for k in ["xlsr_wsum_mlp", "text_tfidf", "text_phobert", "acoustic_ltas", "acoustic_mfcc"]:
    PRED[k] = np.load(f"{RES_DIR}/pred_{k}.npy")
for k, v in PRED.items():
    assert len(v) == n_te, (k, len(v), n_te)

# người nói trong tập test
spk_te = meta.loc[te, "spk"].astype(str).values
SPK, spk_id = np.unique(spk_te, return_inverse=True); K = len(SPK)
cnt = np.bincount(spk_id)
print(f"{C} tỉnh | test: {n_te} câu, {K} người nói | câu/người: trung bình {cnt.mean():.2f}, tối đa {cnt.max()}, "
      f"{(cnt > 1).mean():.0%} người có >1 câu")
REPORT = dict(n_test=int(n_te), n_test_speakers=int(K), utt_per_speaker=dict(mean=round(float(cnt.mean()), 3), max=int(cnt.max()),
              share_multi=round(float((cnt > 1).mean()), 3)))

## 0. Kiểm tra khớp với notebook 10

In [ ]:
ref = {r["model"]: r for r in json.load(open(f"{FIN}/final_report.json"))["table"]}
name_map = {"B_xlsr|ls": "B_xlsr|ls", "C_phowhisper|ls": "C_phowhisper|ls", "D_fusion|ls": "D_fusion|ls",
            "D_fusion|geols": "D_fusion|geols", "ensemble_BCD|ls": "FINAL_acc: B+C+D|ls"}
for k, rk in name_map.items():
    a = float((PRED[k] == yt).mean())
    ok = abs(a - ref[rk]["test_acc"]) < 1e-3
    print(f"{k:18s} acc {a:.4f} | notebook 10: {ref[rk]['test_acc']:.4f} {'OK' if ok else 'KHÔNG KHỚP – kiểm tra lại file'}")

## 1. Bootstrap theo người nói so với bootstrap theo câu

Bootstrap theo câu giả định các câu độc lập. Nếu một người nói có nhiều câu, các câu đó tương quan với nhau, nên khoảng tin cậy có thể bị hẹp quá.
Bootstrap theo người nói lấy mẫu lại **người nói** (kèm toàn bộ câu của họ), giữ nguyên cặp so sánh giữa hai mô hình.

In [ ]:
rng = np.random.default_rng(0)
B_UTT = rng.integers(0, n_te, (N_BOOT, n_te))
B_SPK = rng.integers(0, K, (N_BOOT, K))
spk_n = np.bincount(spk_id, minlength=K).astype(float)
SPK_MEMBERS = [np.where(spk_id == k)[0] for k in range(K)]

def boot_mean(v):
    """Trả về (CI theo câu, CI theo người nói) của trung bình v."""
    s = np.bincount(spk_id, weights=v, minlength=K)
    m_utt = v[B_UTT].mean(1)
    m_spk = s[B_SPK].sum(1) / spk_n[B_SPK].sum(1)
    return np.percentile(m_utt, [2.5, 97.5]), np.percentile(m_spk, [2.5, 97.5]), m_spk

def spk_indices(b):
    return np.concatenate([SPK_MEMBERS[k] for k in B_SPK[b]])

r1 = lambda x: [round(float(v), 4) for v in x]
CI = {}
for k in ["B_xlsr|ls", "C_phowhisper|ls", "D_fusion|ls", "D_fusion|geols", "ensemble_BCD|ls"]:
    ok = (PRED[k] == yt).astype(float); km = DIST[yt, PRED[k]]
    a_u, a_s, _ = boot_mean(ok); k_u, k_s, _ = boot_mean(km)
    f1s = [f1_score(yt[ix], PRED[k][ix], average="macro") for ix in (spk_indices(b) for b in range(500))]
    CI[k] = dict(acc=round(float(ok.mean()), 4), acc_ci_utt=r1(a_u), acc_ci_spk=r1(a_s),
                 macro_f1=round(float(f1_score(yt, PRED[k], average="macro")), 4), macro_f1_ci_spk=r1(np.percentile(f1s, [2.5, 97.5])),
                 km=round(float(km.mean()), 1), km_ci_utt=[round(float(v), 1) for v in k_u], km_ci_spk=[round(float(v), 1) for v in k_s],
                 width_ratio_acc=round(float((a_s[1] - a_s[0]) / (a_u[1] - a_u[0])), 3))
print(pd.DataFrame(CI).T[["acc", "acc_ci_utt", "acc_ci_spk", "width_ratio_acc", "macro_f1", "macro_f1_ci_spk", "km_ci_spk"]].to_string())

PAIRS = [("D_fusion|geols", "D_fusion|ls"), ("D_fusion|ls", "B_xlsr|ls"), ("D_fusion|ls", "C_phowhisper|ls"),
         ("ensemble_BCD|ls", "D_fusion|ls"), ("D_fusion|geols", "xlsr_wsum_mlp")]
PAIRED = {}
for a, b in PAIRS:
    dacc = (PRED[a] == yt).astype(float) - (PRED[b] == yt); dkm = DIST[yt, PRED[a]] - DIST[yt, PRED[b]]
    au, as_, dist_acc = boot_mean(dacc); ku, ks, _ = boot_mean(dkm)
    PAIRED[f"{a} - {b}"] = dict(acc_diff=round(float(dacc.mean()), 4), acc_ci_utt=r1(au), acc_ci_spk=r1(as_),
                                km_diff=round(float(dkm.mean()), 1), km_ci_utt=[round(float(v), 1) for v in ku],
                                km_ci_spk=[round(float(v), 1) for v in ks])
print(json.dumps(PAIRED, indent=1))
REPORT.update(ci=CI, paired=PAIRED)

## 2. Biến thiên theo seed (mô hình kết hợp D, từng seed riêng)

Các khoảng tin cậy ở mục 1 chỉ phản ánh **độ bất định do mẫu test** với một ensemble cố định.
Mục này cho thấy **độ biến thiên do seed huấn luyện**: nạp lại 3 head đã lưu cho LS và Geo-LS, đo từng seed và so sánh từng cặp seed.

In [ ]:
SEED_RES = None
if RUN_SEEDS:
    import torch, torch.nn as nn
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    GROUPS = {s: (g.index.values, g["row"].values) for s, g in meta.groupby("shard")}
    N = len(meta)
    def load_feat(name):
        first = np.load(f"{DRIVE_ROOT}/features/{name}/{next(iter(GROUPS))}.npy")
        X = np.zeros((N,) + first.shape[1:], np.float16)
        for s, (idx, _) in GROUPS.items():
            X[idx] = np.load(f"{DRIVE_ROOT}/features/{name}/{s}.npy")
        return torch.tensor(X, device=device)
    FEATS = {"B": load_feat(FEAT_B), "C": load_feat(FEAT_C)}

    class Head(nn.Module):   # giống notebook 10
        def __init__(s, keys):
            super().__init__(); s.keys = keys
            for k in keys:
                L, Dm = FEATS[k].shape[1:]
                s.register_buffer(f"mu_{k}", torch.zeros(L, Dm)); s.register_buffer(f"sd_{k}", torch.ones(L, Dm))
            s.w = nn.ParameterDict({k: nn.Parameter(torch.zeros(FEATS[k].shape[1])) for k in keys})
            s.mlp = nn.Sequential(nn.Linear(sum(FEATS[k].shape[2] for k in keys), 512), nn.GELU(), nn.Dropout(0.3), nn.Linear(512, C))
        def forward(s, idx):
            parts = []
            for k in s.keys:
                z = (FEATS[k][idx].float() - getattr(s, f"mu_{k}")) / getattr(s, f"sd_{k}")
                parts.append((z * s.w[k].softmax(0)[None, :, None]).sum(1))
            return s.mlp(torch.cat(parts, -1))

    @torch.no_grad()
    def predict(h, mask, bs=1024):
        h.eval(); idx = torch.tensor(np.where(mask)[0], device=device)
        return torch.cat([h(idx[i:i + bs]).softmax(-1) for i in range(0, len(idx), bs)]).cpu().numpy()

    SP = {}
    for loss in ("ls", "geols"):
        for s in SEEDS:
            h = Head(["B", "C"]).to(device)
            h.load_state_dict(torch.load(f"{FIN}/head_D_{loss}_seed{s}.pt", map_location=device))
            SP[(loss, s)] = predict(h, te)
    ens_chk = {l: float((np.mean([SP[(l, s)] for s in SEEDS], 0).argmax(1) == yt).mean()) for l in ("ls", "geols")}
    print("Kiểm tra: accuracy ensemble từ head đã nạp", ens_chk, "| notebook 10:",
          {l: ref[f"D_fusion|{l}"]["test_acc"] for l in ("ls", "geols")})
    rows = []
    for s in SEEDS:
        pl, pg = SP[("ls", s)].argmax(1), SP[("geols", s)].argmax(1)
        dacc = (pg == yt).astype(float) - (pl == yt); dkm = DIST[yt, pg] - DIST[yt, pl]
        _, ci_a, _ = boot_mean(dacc); _, ci_k, _ = boot_mean(dkm)
        rows.append(dict(seed=s, acc_ls=round(float((pl == yt).mean()), 4), acc_geols=round(float((pg == yt).mean()), 4),
                         km_ls=round(float(DIST[yt, pl].mean()), 1), km_geols=round(float(DIST[yt, pg].mean()), 1),
                         acc_diff=round(float(dacc.mean()), 4), acc_diff_ci_spk=r1(ci_a),
                         km_diff=round(float(dkm.mean()), 1), km_diff_ci_spk=[round(float(v), 1) for v in ci_k]))
    SEED_RES = pd.DataFrame(rows)
    print(SEED_RES.to_string(index=False))
    print("Trung bình ± sd qua seed: Δacc", round(SEED_RES.acc_diff.mean(), 4), "±", round(SEED_RES.acc_diff.std(ddof=1), 4),
          "| Δkm", round(SEED_RES.km_diff.mean(), 1), "±", round(SEED_RES.km_diff.std(ddof=1), 1))
    REPORT["seed_level_D"] = dict(runs=SEED_RES.to_dict(orient="records"), ensemble_check=ens_chk)

## 3. Hiệu chuẩn: ECE, Brier, NLL

NLL thấp hơn chưa đủ để nói "hiệu chuẩn tốt hơn". ECE (15 bin, theo nhãn dự đoán) đo độ lệch giữa độ tự tin và độ chính xác; Brier đo sai số bình phương của cả vector xác suất.

In [ ]:
def calib(P, yy, bins=15):
    conf = P.max(1); pred = P.argmax(1); ok = (pred == yy).astype(float)
    edges = np.linspace(0, 1, bins + 1); ece = 0.0; rel = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            ece += m.mean() * abs(ok[m].mean() - conf[m].mean()); rel.append((float(conf[m].mean()), float(ok[m].mean()), int(m.sum())))
    Y = np.eye(P.shape[1])[yy]
    return dict(ece=round(float(ece), 4), brier=round(float(((P - Y) ** 2).sum(1).mean()), 4),
                nll=round(float(-np.log(np.clip(P[np.arange(len(yy)), yy], 1e-12, 1)).mean()), 4),
                mean_conf=round(float(conf.mean()), 4), acc=round(float(ok.mean()), 4)), rel

CAL, REL = {}, {}
for k in ["D_fusion|ls", "D_fusion|geols", "ensemble_BCD|ls", "B_xlsr|ls", "C_phowhisper|ls"]:
    CAL[k], REL[k] = calib(PROBS[k], yt)
if SEED_RES is not None:
    for l in ("ls", "geols"):
        cs = [calib(SP[(l, s)], yt)[0] for s in SEEDS]
        CAL[f"D_fusion|{l} (từng seed, trung bình)"] = {m: round(float(np.mean([c[m] for c in cs])), 4) for m in cs[0]}
print(pd.DataFrame(CAL).T.to_string())

# CI theo người nói cho hiệu ECE và Brier (Geo-LS − LS, mô hình D ensemble)
dE, dB = [], []
for b in range(500):
    ix = spk_indices(b)
    cg, _ = calib(PROBS["D_fusion|geols"][ix], yt[ix]); cl, _ = calib(PROBS["D_fusion|ls"][ix], yt[ix])
    dE.append(cg["ece"] - cl["ece"]); dB.append(cg["brier"] - cl["brier"])
CAL_DIFF = dict(ece_diff=round(CAL["D_fusion|geols"]["ece"] - CAL["D_fusion|ls"]["ece"], 4), ece_diff_ci_spk=r1(np.percentile(dE, [2.5, 97.5])),
                brier_diff=round(CAL["D_fusion|geols"]["brier"] - CAL["D_fusion|ls"]["brier"], 4), brier_diff_ci_spk=r1(np.percentile(dB, [2.5, 97.5])))
print("Geo-LS − LS (D):", CAL_DIFF)
REPORT.update(calibration=CAL, calibration_diff_D=CAL_DIFF)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(4.2, 3.6))
ax.plot([0, 1], [0, 1], color="#999", lw=1, ls="--")
for k, col, lab in [("D_fusion|ls", "#2a78d6", "Fusion, LS"), ("D_fusion|geols", "#eb6834", "Fusion, Geo-LS")]:
    r = np.array(REL[k]); ax.plot(r[:, 0], r[:, 1], marker="o", color=col, lw=1.5, label=f"{lab} (ECE {CAL[k]['ece']:.3f})")
ax.set_xlabel("Confidence"); ax.set_ylabel("Accuracy"); ax.legend(frameon=False, fontsize=8); ax.set_title("Reliability diagram", fontsize=9, loc="left")
fig.tight_layout(); fig.savefig(f"{OUT}/fig_reliability.png", dpi=200); plt.show()

## 4. Không thua kém (non-inferiority) và tương đương: Geo-LS so với LS trên mô hình D

"Khác biệt không có ý nghĩa thống kê" **không** có nghĩa là "bằng nhau". Ở đây dùng bootstrap theo người nói:
- **Không thua kém với biên δ:** cận dưới một phía 95% (phân vị 5%) của Δacc lớn hơn −δ.
- **Tương đương (TOST) với biên δ:** khoảng 90% của Δacc nằm trọn trong (−δ, +δ).

In [ ]:
dacc = (PRED["D_fusion|geols"] == yt).astype(float) - (PRED["D_fusion|ls"] == yt)
_, _, dist_d = boot_mean(dacc)
lo5, lo95 = np.percentile(dist_d, 5), np.percentile(dist_d, 95)
NI = dict(acc_diff=round(float(dacc.mean()), 4), one_sided_lower_95=round(float(lo5), 4), ci90=[round(float(lo5), 4), round(float(lo95), 4)],
          by_margin={f"{m*100:.1f}pt": dict(non_inferior=bool(lo5 > -m), equivalent=bool(lo5 > -m and lo95 < m)) for m in NI_MARGINS})
print(json.dumps(NI, indent=1))
REPORT["non_inferiority_geols_vs_ls_D"] = NI

## 5. Kiểm định Mantel/QAP: nhầm lẫn có theo khoảng cách địa lý không?

Các cặp tỉnh dùng chung tỉnh nên không độc lập; p-value của Spearman thông thường không hợp lệ.
Cách đúng: giữ nguyên ma trận nhầm lẫn, **hoán vị đồng thời hàng và cột** của ma trận khoảng cách (gán ngẫu nhiên vị trí cho các tỉnh), tính lại ρ, lặp lại nhiều lần.
Đơn vị phân tích: **cặp tỉnh không thứ tự**; nhầm lẫn được chuẩn hóa theo số câu của tỉnh thật rồi lấy đối xứng.
Lưu ý: Geo-LS được huấn luyện để nhầm sang tỉnh gần, nên bằng chứng "giọng thay đổi liên tục theo địa lý" phải lấy từ mô hình **CE** (baseline notebook 02).

In [ ]:
iu = np.triu_indices(C, 1)
def sym_rate(pred):
    M = np.zeros((C, C)); np.add.at(M, (yt, pred), 1)
    R = M / np.maximum(M.sum(1, keepdims=True), 1); np.fill_diagonal(R, 0)
    return (R + R.T) / 2

def mantel(S, n_perm=N_PERM, seed=0):
    rs = rankdata(S[iu]); obs = spearmanr(rs, DIST[iu])[0]
    g = np.random.default_rng(seed); null = np.empty(n_perm)
    for i in range(n_perm):
        p = g.permutation(C); null[i] = spearmanr(rs, DIST[p][:, p][iu])[0]
    pval = (1 + (np.abs(null) >= abs(obs)).sum()) / (1 + n_perm)
    return obs, pval, null

MANTEL = {}
for k in ["xlsr_wsum_mlp", "D_fusion|ls", "D_fusion|geols"]:
    S = sym_rate(PRED[k]); rho, pv, null = mantel(S)
    M = np.zeros((C, C)); np.add.at(M, (yt, PRED[k]), 1); off = ~np.eye(C, dtype=bool)
    MANTEL[k] = dict(rho_mantel=round(float(rho), 4), p_perm=float(pv), null_95=r1(np.percentile(null, [2.5, 97.5])),
                     rho_naive_ordered_pairs=round(float(spearmanr(M[off], DIST[off])[0]), 4))
    print(k, MANTEL[k])
    if k == "xlsr_wsum_mlp":
        NULL_CE, RHO_CE = null, rho
REPORT["mantel"] = MANTEL

fig, ax = plt.subplots(figsize=(4.2, 3.0))
ax.hist(NULL_CE, bins=60, color="#bbb"); ax.axvline(RHO_CE, color="#2a78d6", lw=2)
ax.set_xlabel("Spearman ρ under permutation"); ax.set_title(f"Mantel test, CE baseline: ρ = {RHO_CE:.3f}", fontsize=9, loc="left")
fig.tight_layout(); fig.savefig(f"{OUT}/fig_mantel_null.png", dpi=200); plt.show()

## 6. Giới tính

Giới tính chỉ có 2 giá trị nên **tự nó** không thể đoán đúng nhiều tỉnh. Câu hỏi đúng là: (a) tỉ lệ nam/nữ có lệch theo tỉnh không, (b) đoán tỉnh chỉ từ giới tính đạt tối đa bao nhiêu, (c) accuracy của mô hình có khác nhau giữa nam và nữ không.

In [ ]:
g_raw = meta["gender"].astype(str).str.strip().str.lower()
GMAP = {"1": "male", "1.0": "male", "m": "male", "male": "male", "nam": "male",
        "0": "female", "0.0": "female", "f": "female", "female": "female", "nữ": "female", "nu": "female"}
gender = g_raw.map(GMAP).fillna("unknown").values
print("Phân bố giới tính (toàn bộ):", pd.Series(gender).value_counts().to_dict())
tab = pd.crosstab(pd.Series(y[tr], name="province"), pd.Series(gender[tr], name="gender"))
tab = tab[[c for c in ("female", "male") if c in tab.columns]]
GENDER = dict(counts={s: pd.Series(gender[m]).value_counts().to_dict() for s, m in (("train", tr), ("valid", va), ("test", te))})
if tab.shape[1] == 2:
    chi2, pchi, dof, _ = chi2_contingency(tab.values)
    share = tab["male"] / tab.sum(1)
    GENDER.update(chi2=round(float(chi2), 1), dof=int(dof), p=float(pchi),
                  male_share_by_province=dict(min=round(float(share.min()), 3), median=round(float(share.median()), 3), max=round(float(share.max()), 3)))
# (b) giới hạn trên khi chỉ biết giới tính: với mỗi giới, đoán tỉnh xuất hiện nhiều nhất trong train
g_te = gender[te]
best_prov = {g: int(pd.Series(y[tr][gender[tr] == g]).value_counts().idxmax()) for g in np.unique(gender[tr])}
gender_only_pred = np.array([best_prov.get(g, 0) for g in g_te])
GENDER["gender_only_test_acc"] = round(float((gender_only_pred == yt).mean()), 4)
# (c) accuracy theo giới tính
by_g = {}
for k in ["xlsr_wsum_mlp", "D_fusion|ls", "D_fusion|geols", "ensemble_BCD|ls"]:
    ok = (PRED[k] == yt).astype(float); r = {}
    for g in ("male", "female"):
        m = g_te == g
        if m.any():
            r[g] = dict(n=int(m.sum()), acc=round(float(ok[m].mean()), 4))
    if len(r) == 2:   # CI theo người nói cho hiệu nam − nữ
        v = np.where(g_te == "male", 1.0, 0.0)
        sm = np.bincount(spk_id, weights=ok * v, minlength=K); nm = np.bincount(spk_id, weights=v, minlength=K)
        sf = np.bincount(spk_id, weights=ok * (1 - v), minlength=K); nf = np.bincount(spk_id, weights=1 - v, minlength=K)
        d = sm[B_SPK].sum(1) / nm[B_SPK].sum(1) - sf[B_SPK].sum(1) / nf[B_SPK].sum(1)
        r["male_minus_female"] = round(r["male"]["acc"] - r["female"]["acc"], 4); r["ci_spk"] = r1(np.percentile(d, [2.5, 97.5]))
    by_g[k] = r
GENDER["acc_by_gender"] = by_g
print(json.dumps(GENDER, indent=1, ensure_ascii=False))
REPORT["gender"] = GENDER

## 7. Tập con kiểm soát chặt hơn

"TF-IDF đoán sai" không có nghĩa là câu đó không còn thông tin từ vựng; nó chỉ có nghĩa là **mô hình TF-IDF** không khai thác được.
Ở đây thêm các tập con mà **cả hai** mô hình văn bản đều sai, và **cả hai** mô hình phổ (LTAS, MFCC) đều sai.

In [ ]:
w = {k: PRED[k] != yt for k in ["text_tfidf", "text_phobert", "acoustic_ltas", "acoustic_mfcc"]}
SUB = {"all": np.ones(n_te, bool),
       "both_text_wrong": w["text_tfidf"] & w["text_phobert"],
       "both_spectral_wrong": w["acoustic_ltas"] & w["acoustic_mfcc"],
       "all_four_wrong": w["text_tfidf"] & w["text_phobert"] & w["acoustic_ltas"] & w["acoustic_mfcc"]}
rows = []
for sn, m in SUB.items():
    r = dict(subset=sn, n=int(m.sum()))
    for k in ["xlsr_wsum_mlp", "C_phowhisper|ls", "D_fusion|ls", "D_fusion|geols"]:
        ok = (PRED[k][m] == yt[m]).astype(float); r[k] = round(float(ok.mean()), 4)
        sp = spk_id[m]; Km = np.unique(sp); remap = np.searchsorted(Km, sp)
        s = np.bincount(remap, weights=ok); n = np.bincount(remap).astype(float)
        bb = np.random.default_rng(2).integers(0, len(Km), (1000, len(Km)))
        r[k + " ci_spk"] = r1(np.percentile(s[bb].sum(1) / n[bb].sum(1), [2.5, 97.5]))
    rows.append(r)
SUBT = pd.DataFrame(rows); print(SUBT.to_string(index=False))
REPORT["stricter_subsets"] = SUBT.to_dict(orient="records")

In [ ]:
REPORT["total_minutes"] = round((time.time() - T0) / 60, 1)
json.dump(REPORT, open(f"{OUT}/audit_report.json", "w"), indent=1, ensure_ascii=False, default=float)
print(f"Đã lưu {OUT}/audit_report.json – gửi file này cho Claude. ({REPORT['total_minutes']} phút)")